# Destek 02 — Ham BM dosyaları ile `countries_clean.csv` doğrulaması

**Amaç:** Ekibin tek veri kaynağı `countries_clean.csv`'nin ham BM dosyalarından doğru türetildiğini bağımsız olarak doğrulamak. Rol rehberindeki genel kontrolleri çalıştırır: ülke toplamı Dünya satırına eşit mi, bölge toplamları BM bölge satırlarına eşit mi, doğum toplamları tutuyor mu, ülke sayıları aynı mı.

**Girdi (`data/raw/`, repoda yok, herkes kendi bilgisayarında tutar):**
- `WPP2024_Demographic_Indicators_Medium_csv` (Medium varyantı; ülke, bölge, dünya ve diğer grup satırları)
- `WPP2024_Supplement_Births_and_LE60.csv` (BirthsMale, BirthsFemale, LE60)
- `data/clean/countries_clean.csv`

**Çıktı:** Kontrol tablosu (dosya yazmaz). Grup satırları (bölge, gelir grubu, Dünya) yalnızca kontrol için kullanılır, ülke toplamlarına karıştırılmaz.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW = ROOT / "data/raw"

def find_raw(*keywords):
    hits = sorted(p for p in RAW.glob("*") if p.is_file() and all(k in p.name.lower() for k in keywords))
    assert hits, f"data/raw içinde {keywords} dosyası yok"
    return hits[0]

f_med = find_raw("demographic_indicators_medium"); f_sup = find_raw("supplement")
print("Medium :", f_med.name); print("Supp.  :", f_sup.name)
med = pd.read_csv(f_med, encoding="utf-8-sig", low_memory=False)      # BOM'lu UTF-8
sup = pd.read_csv(f_sup)
clean = pd.read_csv(ROOT / "data/clean/countries_clean.csv")
print("Medium:", med.shape, "| Supplement:", sup.shape, "| clean:", clean.shape)
display(med.groupby("LocTypeName").Location.nunique().to_frame("satır_türü_başına_benzersiz_konum"))
print("Variant değerleri:", med.Variant.unique())

Medium : WPP2024_Demographic_Indicators_Medium_csv
Supp.  : WPP2024_Supplement_Births_and_LE60.csv


Medium: (84360, 67) | Supplement: (35636, 9) | clean: (36024, 68)


,satır_türü_başına_benzersiz_konum
LocTypeName,
Ad Hoc groups,23
Country/Area,237
Development group,7
Geographic region,7
Income group,9
SDG region,9
Special other,7
Subregion,21
World,1


Variant değerleri: <StringArray>
['Medium']
Length: 1, dtype: str


## 1. Temiz dosya ham dosyanın ülke satırlarıyla aynı mı?

Ham Medium'un `Country/Area` satırları + Supplement birleştirilir, `countries_clean.csv` ile her sayısal sütunda karşılaştırılır.

In [2]:
ctry = med[med.LocTypeName == "Country/Area"]
raw_full = ctry.merge(sup.drop(columns=["ISO3_code", "Location"]), on=["LocID", "Time"], how="left")
k = clean.merge(raw_full, on=["LocID", "Time"], suffixes=("_c", "_r"), how="outer", indicator=True)
ortak = [x for x in clean.columns if x in raw_full.columns and x not in ("LocID", "Time") and pd.api.types.is_numeric_dtype(clean[x])]

farkli = {}
for col in ortak:
    a = k[col + "_c"].astype(float).fillna(-9e9); b = k[col + "_r"].astype(float).fillna(-9e9)
    n = int(((a - b).abs() > 1e-6).sum())
    if n: farkli[col] = n

t1 = pd.DataFrame([
 ("Ham ülke satırı (Country/Area)",                  len(ctry), 36024),
 ("Ham ülke sayısı",                                 ctry.LocID.nunique(), 237),
 ("Yalnızca ham ya da yalnızca temizde olan satır",  int((k._merge != "both").sum()), 0),
 ("Karşılaştırılan sayısal sütun",                   len(ortak), len(ortak)),
 ("Farklı çıkan sütun sayısı",                       len(farkli), 0),
 ("Her iki dosyada 2101 satırı",                     int((clean.Time == 2101).sum()), int((ctry.Time == 2101).sum())),
 ("Supplement ülke sayısı (Vatikan yok)",            sup.LocID.nunique(), 236),
], columns=["kontrol", "bulunan", "beklenen"])
t1["tamam"] = t1.bulunan == t1.beklenen
display(t1)
if farkli: print("Farklı sütunlar:", farkli)
print("Supplement'te olmayan ülke:", sorted(set(ctry.Location) - set(sup.Location)))

,kontrol,bulunan,beklenen,tamam
0,Ham ülke satırı (Country/Area),36024,36024,True
1,Ham ülke sayısı,237,237,True
2,Yalnızca ham ya da yalnızca temizde olan satır,0,0,True
3,Karşılaştırılan sayısal sütun,60,60,True
4,Farklı çıkan sütun sayısı,0,0,True
5,Her iki dosyada 2101 satırı,237,237,True
6,Supplement ülke sayısı (Vatikan yok),236,236,True


Supplement'te olmayan ülke: ['Holy See']


## 2. Ülke toplamı = Dünya satırı mı?

Her yıl için ülkelerin toplamı BM'nin `World` satırıyla karşılaştırılır (2101 hariç).

In [3]:
world = med[(med.LocTypeName == "World") & (med.Time <= 2100)].set_index("Time")
cs = clean[clean.Time <= 2100].groupby("Time")[["TPopulation1Jan", "TPopulation1July", "Births", "Deaths", "NatChange", "PopChange", "NetMigrations"]].sum()
rows = []
for col in cs.columns:
    d = (cs[col] - world[col]).abs()
    rel = (d / world[col].abs().replace(0, np.nan)).max()
    rows.append((col, round(d.max(), 3), rel, int(d.idxmax())))
t2 = pd.DataFrame(rows, columns=["değişken", "en_büyük_mutlak_fark_(bin_kişi)", "en_büyük_göreli_fark", "yıl"])
t2["ölçüt"] = np.where(t2.değişken == "NetMigrations", "mutlak < 10 bin (Dünya'da net göç 0)", "göreli < 1e-4")
t2["tamam"] = np.where(t2.değişken == "NetMigrations", t2.iloc[:, 1] < 10, t2.en_büyük_göreli_fark.fillna(0) < 1e-4)
display(t2)
print("Dünya nüfusu zirve:", int(world.TPopulation1July.idxmax()), "→", round(world.TPopulation1July.max() / 1e6, 3), "milyar")

,değişken,en_büyük_mutlak_fark_(bin_kişi),en_büyük_göreli_fark,yıl,ölçüt,tamam
0,TPopulation1Jan,0.000,1.958281e-16,2047,göreli < 1e-4,True
1,TPopulation1July,0.014,3.609974e-09,2024,göreli < 1e-4,True
2,Births,0.021,1.730926e-07,1976,göreli < 1e-4,True
3,Deaths,0.022,3.633670e-07,2043,göreli < 1e-4,True
4,NatChange,0.030,2.431756e-05,2031,göreli < 1e-4,True
5,PopChange,0.000,4.133258e-15,1984,göreli < 1e-4,True
6,NetMigrations,4.989,NaN,2049,mutlak < 10 bin (Dünya'da net göç 0),True


Dünya nüfusu zirve: 2084 → 10.289 milyar


**Okuma:** Ülke toplamı Dünya satırından yalnızca yuvarlama düzeyinde ayrılıyor (nüfusta en çok birkaç yüz kişi). Net göçte fark birkaç bin kişiye kadar çıkıyor: Dünya için net göç tanım gereği sıfırdır, ülke değerlerinin toplamı ise yuvarlama nedeniyle tam sıfır olmuyor. Bu bir hata değil, birimi bin kişi olan verinin yuvarlama gürültüsü.

## 3. Bölge toplamları BM bölge satırlarıyla aynı mı?

In [4]:
reg = med[(med.LocTypeName == "Geographic region") & (med.Time <= 2100)]
un = reg.pivot(index="Time", columns="Location", values="TPopulation1July")
mine = clean[clean.Time <= 2100].groupby(["Time", "Region"]).TPopulation1July.sum().unstack()
rows = []
for r in mine.columns:
    d = (mine[r] - un[r]).abs(); rows.append((r, round(d.max(), 3), (d / un[r]).max()))
t3 = pd.DataFrame(rows, columns=["bölge", "en_büyük_mutlak_fark_(bin_kişi)", "en_büyük_göreli_fark"])
t3["tamam"] = t3.en_büyük_göreli_fark < 1e-6
display(t3)
am = (un["Americas"] - un["Latin America and the Caribbean"] - un["Northern America"]).abs().max()
print(f"BM 'Americas' satırı = Latin Amerika + Kuzey Amerika (en büyük fark {am:.3f} bin kişi); analizlerde Americas kullanılmaz.")
print("Bölge başına ülke sayısı (clean):", clean.groupby("Region").LocID.nunique().to_dict())

,bölge,en_büyük_mutlak_fark_(bin_kişi),en_büyük_göreli_fark,tamam
0,Africa,0.007,1.756108e-08,True
1,Asia,0.008,5.847631e-09,True
2,Europe,0.011,1.821766e-08,True
3,Latin America and the Caribbean,0.008,3.281172e-08,True
4,Northern America,0.002,1.098906e-08,True
5,Oceania,0.004,2.267753e-07,True


BM 'Americas' satırı = Latin Amerika + Kuzey Amerika (en büyük fark 0.001 bin kişi); analizlerde Americas kullanılmaz.
Bölge başına ülke sayısı (clean): {'Africa': 58, 'Asia': 51, 'Europe': 50, 'Latin America and the Caribbean': 50, 'Northern America': 5, 'Oceania': 23}


## 4. Doğum toplamları ve göç tutarlılığı (ham dosyada)

In [5]:
x = raw_full.dropna(subset=["BirthsMale"])
fark_dogum = (x.BirthsMale + x.BirthsFemale - x.Births).abs()
r_ = (raw_full.NatChange + raw_full.NetMigrations - raw_full.PopChange).abs()
t4 = pd.DataFrame([
 ("BirthsMale+BirthsFemale ≠ Births (>0.01 bin)",       int((fark_dogum > 0.01).sum()), 0),
 ("NatChange+NetMigrations ≠ PopChange (>5 bin)",       int((r_ > 5).sum()), 0),
 ("NatChange+NetMigrations ≠ PopChange (>1 bin) [bilgi]", int((r_ > 1).sum()), "yuvarlama"),
 ("161 büyük ülke (2024 nüfusu > 1 milyon)",            int((clean[clean.Time == 2024].TPopulation1July > 1000).sum()), 161),
], columns=["kontrol", "bulunan", "beklenen"])
_ulk = ROOT / "data/clean/ulkeler_1m.csv"
if _ulk.exists():
    _u = pd.read_csv(_ulk); _kol = {k.lower(): k for k in _u.columns}
    _hesap = set(clean[(clean.Time == 2024) & (clean.TPopulation1July > 1000)].ISO3_code)
    t4.loc[len(t4)] = ("ulkeler_1m.csv ISO3 listesi = hesaplanan 161 (1 = aynı)", int(set(_u[_kol["iso3"]]) == _hesap), 1)
else:
    print("ulkeler_1m.csv data/clean içinde yok; liste karşılaştırması atlandı.")
display(t4)

,kontrol,bulunan,beklenen
0,BirthsMale+BirthsFemale ≠ Births (>0.01 bin),0,0
1,NatChange+NetMigrations ≠ PopChange (>5 bin),0,0
2,NatChange+NetMigrations ≠ PopChange (>1 bin) [...,10,yuvarlama
3,161 büyük ülke (2024 nüfusu > 1 milyon),161,161
4,ulkeler_1m.csv ISO3 listesi = hesaplanan 161 (...,1,1


## 5. Sonuç

- `countries_clean.csv`, ham Medium (ülke satırları) ve Supplement'ten **birebir** türetilmiş: tüm sayısal sütunlarda fark yok.
- Ülke toplamı Dünya satırına ve bölge toplamları BM bölge satırlarına yuvarlama düzeyinde eşit.
- `Region` ataması BM'nin bölge tanımıyla uyumlu (6 bölge; 7. satır "Americas" ikisinin toplamı).
- Grup satırları (bölge, gelir grubu, Dünya) yalnızca kontrol ve medyan yaş (g01 B) için kullanıldı, ülke toplamlarına karıştırılmadı.

Kontrol listesine etkisi: "ülke toplamı Dünya satırına eşit mi?" ve "237 / 161 ülke sayıları tüm dosyalarda aynı mı?" genel kontrolleri geçti.